In [9]:
import pyspark
from pyspark.sql import SparkSession

In [10]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework') \
    .getOrCreate()


# QUESTION 1 - spark version

In [11]:
print(f"Spark version: {spark.version}")

Spark version: 4.1.1


In [12]:
df = spark.read.parquet("data/yellow_tripdata_2025-11.parquet")

In [13]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [14]:
df.head(5)

[Row(VendorID=7, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 0, 13, 25), passenger_count=1, trip_distance=1.68, RatecodeID=1, store_and_fwd_flag='N', PULocationID=43, DOLocationID=186, payment_type=1, fare_amount=14.9, extra=0.0, mta_tax=0.5, tip_amount=1.5, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.15, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 1, 0, 49, 7), tpep_dropoff_datetime=datetime.datetime(2025, 11, 1, 1, 1, 22), passenger_count=1, trip_distance=2.28, RatecodeID=1, store_and_fwd_flag='N', PULocationID=142, DOLocationID=237, payment_type=1, fare_amount=14.2, extra=1.0, mta_tax=0.5, tip_amount=4.99, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=24.94, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2025, 11, 1,

In [16]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

# QUESTION 2 - 4 partitioned avg size

In [17]:
df.repartition(4).write.mode('overwrite').parquet('data/repartitioned_taxi')


# QUESTION 3 - trip count 

In [18]:
from pyspark.sql import functions as F

In [ ]:
# display top five on 3035 nov 15
df.filter(F.to_date(df.tpep_pickup_datetime) == "2025-11-15").head(5)

[Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 15, 0, 0, 16), tpep_dropoff_datetime=datetime.datetime(2025, 11, 15, 0, 7, 22), passenger_count=2, trip_distance=1.27, RatecodeID=1, store_and_fwd_flag='N', PULocationID=137, DOLocationID=113, payment_type=1, fare_amount=8.6, extra=1.0, mta_tax=0.5, tip_amount=2.87, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=17.22, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2025, 11, 15, 0, 3, 1), tpep_dropoff_datetime=datetime.datetime(2025, 11, 15, 0, 29, 53), passenger_count=1, trip_distance=4.25, RatecodeID=1, store_and_fwd_flag='N', PULocationID=249, DOLocationID=237, payment_type=1, fare_amount=26.1, extra=1.0, mta_tax=0.5, tip_amount=3.18, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=35.03, congestion_surcharge=2.5, Airport_fee=0.0, cbd_congestion_fee=0.75),
 Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2025, 11,

In [21]:
print("Num trips on 2025 nov 15:", df.filter(F.to_date(df.tpep_pickup_datetime) == "2025-11-15").count())

Num trips on 2025 nov 15: 162604



# QUESTION 4 - longest trip

In [22]:
# creating a temp view to query using SQL
df.createOrReplaceTempView("yellow_taxi_trips")

In [31]:
spark.sql('\
    SELECT MAX(timestampdiff(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime) / 3600.0) AS longest_trip_duration_hours \
    FROM yellow_taxi_trips;' ).show()

+---------------------------+
|longest_trip_duration_hours|
+---------------------------+
|                  90.646667|
+---------------------------+



# QUESTION 5 - spark app ui local server port
* it runs in the host port __4040__ by default (if no other apps are hosted on that port)